In [0]:
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")

CATALOG       = dbutils.widgets.get("catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

# Safe to run standalone: make sure the silver schema exists first
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
# Small throwaway table to see schema enforcement vs evolution safely
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.schema_evo_demo")
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SILVER_SCHEMA}.schema_evo_demo (
        id     BIGINT,
        label  STRING
    ) USING DELTA
""")
spark.sql(f"INSERT INTO {CATALOG}.{SILVER_SCHEMA}.schema_evo_demo VALUES (1,'a'),(2,'b')")

display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.schema_evo_demo ORDER BY id"))

In [0]:
# A new batch carries an EXTRA column ('score') the table does not have yet.
# Default Delta behavior: reject the write (schema enforcement).
new_df = spark.createDataFrame([(3, "c", 99)], ["id", "label", "score"])

try:
    (new_df.write.format("delta").mode("append")
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.schema_evo_demo"))
    print("Appended (unexpected - enforcement did not fire)")
except Exception as e:
    print("REJECTED by schema enforcement (expected):\n")
    print(str(e)[:300])

In [0]:
# Same write, now we opt in to letting the schema grow.
(new_df.write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.schema_evo_demo"))

display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.schema_evo_demo ORDER BY id"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER_SCHEMA}.schema_evo_demo")